<a href="https://colab.research.google.com/github/pesco13/Python-para-Quimica/blob/main/notebooks/04_Calibracion_y_error.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

# 04 · Curva de calibración y error experimental

**Objetivos**
* Ajustar una recta a datos de calibración y entender qué significa cada número del ajuste.
* Calcular la concentración de una muestra problema **con su incertidumbre**.
* Obtener el límite de detección y el de cuantificación.
* Leer una gráfica de residuales y reconocer cuándo el ajuste no sirve.

Esta es la práctica más parecida a lo que se hace en el laboratorio de análisis: se miden patrones, se ajusta una recta y se usa esa recta para averiguar cuánto hay en una muestra desconocida.

La siguiente celda descarga el generador desde el sitio del curso y **muestra dónde quedó el archivo**. Debe imprimir una ruta que termine en `generar_datos.py` y un tamaño de unos 8 kB.

El archivo cae en la carpeta de trabajo de la sesión, que en Colab es `/content`. Si abre el icono de carpeta de la barra izquierda y no lo ve, pulse el botón de **refrescar** de ese panel: Colab no lo actualiza solo. Como esa carpeta vive en la máquina temporal, el archivo desaparece al cerrar la sesión y hay que repetir la descarga tras cada reinicio del entorno.


In [ ]:
# La descarga no es silenciosa: si algo falla, wget lo dice.
!wget -O generar_datos.py https://pesco13.github.io/Python-para-Quimica/datos/generar_datos.py

import os

ruta = os.path.abspath("generar_datos.py")
print()
print("Guardado en :", ruta)
print("Tamaño      :", os.path.getsize(ruta), "bytes")
print("La carpeta contiene:", sorted(os.listdir()))

In [ ]:
from generar_datos import calibracion, mi_asignacion

NUMERO = 7          # <<< cambie la semilla para obtener otra serie

print(mi_asignacion(NUMERO))
patrones, problema = calibracion(NUMERO)
patrones.head()

### Mirar la tabla antes de usarla

Nunca se empieza a calcular sobre una tabla sin haberla mirado. Cuatro comprobaciones bastan: cuántas filas y columnas hay, cómo se llaman las columnas, de qué tipo es cada una y qué valores toman. Si una columna de números aparece como `object`, es que llegó como texto y hay que convertirla antes de operar.

In [ ]:
# Lo primero con cualquier tabla: mirarla.
print("Filas y columnas:", patrones.shape)
print("Columnas        :", patrones.columns.tolist())
print()
print("Tipo de dato de cada columna:")
print(patrones.dtypes)
print()
print("Primeras filas:")
print(patrones.head())
print()
print("Resumen numérico:")   # cuenta, media, desviación, mínimo, máximo y cuartiles
patrones.describe()

`patrones` tiene seis niveles de concentración medidos por triplicado; `problema` tiene tres réplicas de la muestra desconocida.

In [ ]:
print(problema)
print("\nNúmero de medidas de patrones:", len(patrones))

## 1. La ley de Beer–Lambert

La absorbancia de una disolución diluida es proporcional a la concentración:

$$A = \varepsilon \, \ell \, c$$

donde $\varepsilon$ es la absortividad molar, $\ell$ el paso óptico de la celda (1 cm casi siempre) y $c$ la concentración. En la práctica se ajusta

$$A = m\,c + b$$

y se usa la recta para invertir el cálculo. El intercepto $b$ debería ser cero, pero nunca lo es del todo: el blanco, la celda y la deriva del equipo dejan un pequeño residuo.

## 2. Promediar las réplicas y mirar los datos

Antes de ajustar nada, mire los puntos.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

resumen = patrones.groupby("concentracion_mg_L")["absorbancia"].agg(["mean", "std", "count"])
resumen

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.errorbar(resumen.index, resumen["mean"], yerr=resumen["std"],
            fmt="o", capsize=4, color="#1b6ca8")
ax.set_xlabel("Concentración (mg L$^{-1}$)")
ax.set_ylabel("Absorbancia")
ax.set_title("Patrones medidos por triplicado")
plt.show()

## 3. El ajuste de la recta

`scipy.stats.linregress` ajusta por mínimos cuadrados. Se ajusta sobre **todas** las medidas individuales, no sobre los promedios: así el ajuste tiene en cuenta cuánto dispersa cada nivel.

In [ ]:
from scipy.stats import linregress

x = patrones["concentracion_mg_L"].values
y = patrones["absorbancia"].values

ajuste = linregress(x, y)

print(f"Pendiente   m = {ajuste.slope:.5f} ± {ajuste.stderr:.5f} (A por mg/L)")
print(f"Intercepto  b = {ajuste.intercept:.5f} ± {ajuste.intercept_stderr:.5f}")
print(f"Coeficiente r = {ajuste.rvalue:.5f}   ->  R² = {ajuste.rvalue**2:.5f}")

### Qué dice y qué no dice el R²

Un R² de 0,999 **no** garantiza que la recta sea el modelo correcto. El R² mide cuánta de la variación total explica el modelo, y con datos de calibración casi siempre sale altísimo porque el intervalo de concentraciones es ancho. Una curva suave disfrazada de recta puede dar R² = 0,9995 y aun así estar sesgando todos los resultados.

Lo que sí detecta el problema es la gráfica de **residuales**.

## 4. Residuales

El residual es la diferencia entre lo medido y lo que predice la recta:

$$e_i = y_i - (m x_i + b)$$

Si el modelo es correcto, los residuales se reparten al azar arriba y abajo del cero. Si forman una curva, una U o un embudo, el modelo lineal no describe los datos.

In [ ]:
prediccion = ajuste.slope * x + ajuste.intercept
residuales = y - prediccion

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(6, 6), sharex=True,
                               gridspec_kw={"height_ratios": [2, 1]})

ax1.plot(x, y, "o", color="#1b6ca8", alpha=0.7, label="medidas")
xx = np.linspace(x.min(), x.max(), 100)
ax1.plot(xx, ajuste.slope * xx + ajuste.intercept, "-", color="#d1495b",
         label=f"$A$ = {ajuste.slope:.4f}·$c$ + {ajuste.intercept:.4f}")
ax1.set_ylabel("Absorbancia")
ax1.legend(frameon=False)
ax1.set_title("Ajuste y residuales de la calibración")

ax2.axhline(0, color="0.5", lw=0.8)
ax2.plot(x, residuales, "o", color="#1b6ca8", alpha=0.7)
ax2.set_xlabel("Concentración (mg L$^{-1}$)")
ax2.set_ylabel("Residual")

fig.tight_layout()
fig.savefig("calibracion_residuales.png", dpi=300, bbox_inches="tight")
plt.show()

Mire el panel de abajo con atención. En estos datos los residuales del nivel más alto tienden a quedar por debajo del cero: la respuesta se va aplanando, como pasa de verdad cuando la absorbancia se acerca a 1 y la ley de Beer deja de cumplirse. Eso es información, no ruido.

Más adelante, en los ejercicios, se le pide repetir el ajuste quitando el nivel más alto y comparar.

## 5. La desviación estándar del ajuste

El número que resume cuánto se alejan los puntos de la recta es la desviación estándar residual:

$$s_{y/x} = \sqrt{\frac{\sum (y_i - \hat{y}_i)^2}{n - 2}}$$

Se divide entre $n-2$ porque se gastaron dos grados de libertad en estimar la pendiente y el intercepto.

In [ ]:
n = len(x)
s_yx = np.sqrt(np.sum(residuales**2) / (n - 2))
Sxx = np.sum((x - x.mean())**2)

print(f"s_y/x = {s_yx:.5f} unidades de absorbancia")
print(f"n     = {n}")

## 6. La muestra problema y su incertidumbre

Se despeja la concentración de la recta:

$$c_0 = \frac{\bar{A}_0 - b}{m}$$

y la incertidumbre de ese valor tiene tres contribuciones: el ruido de las réplicas de la muestra, el ruido de los patrones y lo lejos que esté la muestra del centro de la calibración:

$$s_{c_0} = \frac{s_{y/x}}{m} \sqrt{\frac{1}{M} + \frac{1}{n} + \frac{(\bar{A}_0 - \bar{y})^2}{m^2 S_{xx}}}$$

con $M$ el número de réplicas de la muestra y $S_{xx} = \sum (x_i - \bar{x})^2$.

In [ ]:
A0 = problema["absorbancia"].mean()
M = len(problema)

c0 = (A0 - ajuste.intercept) / ajuste.slope
s_c0 = (s_yx / ajuste.slope) * np.sqrt(1/M + 1/n + (A0 - y.mean())**2 / (ajuste.slope**2 * Sxx))

print(f"Absorbancia media de la muestra: {A0:.4f}")
print(f"Concentración: {c0:.3f} ± {s_c0:.3f} mg/L  (desviación estándar)")

Para dar un **intervalo de confianza del 95 %** hay que multiplicar por el valor $t$ de Student con $n-2$ grados de libertad, no por 1,96: con pocos datos la distribución es más ancha que la normal.

In [ ]:
from scipy.stats import t as t_student

t_critico = t_student.ppf(0.975, df=n - 2)
margen = t_critico * s_c0

print(f"t crítico (95 %, {n-2} gl) = {t_critico:.3f}")
print(f"Concentración = {c0:.2f} ± {margen:.2f} mg/L  (IC 95 %)")
print(f"Es decir, entre {c0-margen:.2f} y {c0+margen:.2f} mg/L")

### Cifras significativas

La incertidumbre manda. Si el margen es 0,34 mg/L, no tiene sentido escribir la concentración con cuatro decimales: se redondea la incertidumbre a una o dos cifras significativas y el resultado se reporta con el mismo número de decimales.

In [ ]:
def reportar(valor, incertidumbre):
    """Redondea el resultado al mismo decimal que la incertidumbre."""
    if incertidumbre <= 0:
        return f"{valor}"
    decimales = max(0, -int(np.floor(np.log10(incertidumbre))))
    return f"{valor:.{decimales}f} ± {incertidumbre:.{decimales}f}"

print("Resultado final:", reportar(c0, margen), "mg/L")

## 7. Límite de detección y de cuantificación

El límite de detección es la concentración más baja que se distingue del ruido de fondo; el de cuantificación, la más baja que se puede medir con precisión aceptable. El criterio habitual (IUPAC) usa la desviación del blanco o, si no hay suficientes blancos, $s_{y/x}$:

$$\text{LOD} = \frac{3 s_{y/x}}{m} \qquad \text{LOQ} = \frac{10 s_{y/x}}{m}$$

In [ ]:
LOD = 3 * s_yx / ajuste.slope
LOQ = 10 * s_yx / ajuste.slope

print(f"LOD = {LOD:.3f} mg/L")
print(f"LOQ = {LOQ:.3f} mg/L")
print("¿La muestra está por encima del LOQ?", "sí" if c0 > LOQ else "no")

## Autocomprobación

In [ ]:
assert 0.0 < ajuste.slope < 1.0, "la pendiente debe ser positiva y del orden de 0,1 A por mg/L"
assert abs(c0 - (A0 - ajuste.intercept) / ajuste.slope) < 1e-9, "revise el despeje de c0"
assert s_c0 > 0 and margen > s_c0, "el margen al 95 % tiene que ser mayor que una desviación estándar"
assert LOQ > LOD > 0, "el LOQ siempre es mayor que el LOD"
print("Todo correcto.")

## Ejercicios

**1.** Repita el ajuste usando solo los cinco niveles más bajos de concentración. ¿Cambia la pendiente? ¿Mejora la gráfica de residuales?

<details><summary><b>Ver solución</b></summary>

```python
sub = patrones[patrones["concentracion_mg_L"] < patrones["concentracion_mg_L"].max()]
ajuste2 = linregress(sub["concentracion_mg_L"], sub["absorbancia"])
print(ajuste.slope, "->", ajuste2.slope)
```
La pendiente aumenta un poco, porque el nivel alto tiraba la recta hacia abajo. Es el intervalo lineal real del método.
</details>

**2.** Calcule la concentración de la muestra con la recta del ejercicio 1 y compare con el resultado original. ¿La diferencia es mayor o menor que la incertidumbre?

<details><summary><b>Ver solución</b></summary>

```python
c0_b = (A0 - ajuste2.intercept) / ajuste2.slope
print(c0, c0_b, abs(c0 - c0_b), margen)
```
Si la diferencia es menor que el margen del IC 95 %, las dos rectas dan el mismo resultado dentro del error.
</details>

**3.** ¿Qué pasa con la incertidumbre si la muestra tiene una absorbancia cerca del extremo de la calibración en vez de en el centro? Pruebe con `A0 = 0.05` y con `A0 = ` el valor máximo de los patrones.

<details><summary><b>Ver solución</b></summary>

```python
for prueba in [0.05, y.max()]:
    s = (s_yx / ajuste.slope) * np.sqrt(1/M + 1/n + (prueba - y.mean())**2 / (ajuste.slope**2 * Sxx))
    print(prueba, round(s, 4))
```
La incertidumbre crece hacia los extremos: el término $(\bar{A}_0-\bar{y})^2$ es cero en el centro de la calibración. Por eso conviene diluir la muestra hasta que caiga en la mitad del intervalo.
</details>

**4.** El criterio del LOD usa la desviación del blanco. Calcule la desviación estándar de las tres réplicas del nivel cero y compárela con $s_{y/x}$. ¿Cuál da un LOD más exigente?

<details><summary><b>Ver solución</b></summary>

```python
s_blanco = patrones[patrones["concentracion_mg_L"] == 0]["absorbancia"].std(ddof=1)
print(3 * s_blanco / ajuste.slope, 3 * s_yx / ajuste.slope)
```
</details>